<a href="https://colab.research.google.com/github/paintedpotato/extractsermons/blob/main/I_am_(mentions_by_Pr_EM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# "I am" mentions, past 10 years, per-year progress bars
# Paste each section into its own Colab/Jupyter cell.
# ============================================================

!pip install -q yt-dlp pandas tqdm

In [5]:
##NEW SECTION 1: fetch the video list with upload dates, keep last 10 years
import subprocess, re, bisect
from collections import deque
from datetime import date
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
CHANNEL_URL  = "https://www.youtube.com/channel/UCF9jrwYsvw9UlSvqY6VkQbg"
TABS         = ["videos", "streams"]   # Sunday services are usually under "streams" (past livestreams); add "shorts" if wanted
YEARS_BACK   = 10
OUT_DIR      = "transcripts"
VIDEOS_CSV   = "videos.csv"
TRANSCRIPTS_CSV = "all_transcripts.csv"
LANG         = "en"

today  = date.today()
CUTOFF = today.replace(year=today.year - YEARS_BACK).strftime("%Y%m%d")
# The fast listing only gives approximate dates ("3 years ago"), so section 1 keeps
# one extra year as a buffer. Section 2 replaces them with exact dates and
# section 3 drops anything older than CUTOFF.
BUFFER_CUTOFF = today.replace(year=today.year - YEARS_BACK - 1).strftime("%Y%m%d")

def run(cmd):
    return subprocess.run(cmd, shell=True, stdout=subprocess.PIPE,
                          stderr=subprocess.PIPE, text=True)

def get_videos(channel_url):
    """Fast flat listing. 'approximate_date' makes yt-dlp return an
    upload_date for each entry without opening every video."""
    rows = []
    for tab in TABS:
        print(f"Fetching video list ({tab})...")
        proc = run(
            f'yt-dlp --flat-playlist --extractor-args "youtubetab:approximate_date" '
            f'--print "%(id)s\t%(upload_date)s\t%(title)s" "{channel_url}/{tab}"'
        )
        n = 0
        for line in proc.stdout.splitlines():
            parts = line.split("\t", 2)
            if len(parts) == 3:
                rows.append({"video_id": parts[0], "upload_date": parts[1],
                             "title": parts[2], "tab": tab})
                n += 1
        print(f"  {tab}: {n} entries")
        if proc.returncode != 0 and n == 0:
            print("  (tab missing or failed)", proc.stderr.strip().splitlines()[-1:] )
    df = pd.DataFrame(rows).drop_duplicates("video_id")

    # Any entry without a date: look it up individually (slower, but exact)
    missing = df["upload_date"].isin(["NA", "", None])
    for i in tqdm(df.index[missing], desc="Looking up missing dates", unit="video"):
        p = run(f'yt-dlp --skip-download --print "%(upload_date)s" '
                f'"https://www.youtube.com/watch?v={df.at[i, "video_id"]}"')
        df.at[i, "upload_date"] = p.stdout.strip() or "NA"
    return df

videos = get_videos(CHANNEL_URL)
videos = videos[videos["upload_date"].str.fullmatch(r"\d{8}")]
videos = videos[videos["upload_date"] >= BUFFER_CUTOFF].copy()
videos["year"] = videos["upload_date"].str[:4].astype(int)
videos.to_csv(VIDEOS_CSV, index=False)

print(f"\n{len(videos)} videos since {BUFFER_CUTOFF} (approximate dates, exact ones come in section 2):")
print(videos.pivot_table(index="year", columns="tab", values="video_id",
                         aggfunc="count", fill_value=0).to_string())



Fetching video list (videos)...
  videos: 154 entries
Fetching video list (streams)...
  streams: 309 entries


Looking up missing dates: 0video [00:00, ?video/s]


463 videos since 20151106 (approximate dates, exact ones come in section 2):
tab   streams  videos
year                 
2016        0      18
2017        0      28
2018        0      15
2019        0      18
2020        0      13
2021       19      14
2022       59      10
2023       81      22
2024       52       6
2025       58       5
2026       40       5


In [6]:
##NEW SECTION 2: download captions, one progress bar per year
def download_captions_by_year(videos, out_dir):
    Path(out_dir).mkdir(exist_ok=True)
    totals = {"ok": 0, "skipped": 0, "failed": 0}

    for year, group in sorted(videos.groupby("year"), reverse=True):
        ok = failed = skipped = 0
        bar = tqdm(group.itertuples(), total=len(group),
                   desc=f"{year} captions", unit="video")
        for row in bar:
            vid = row.video_id
            had_file = bool(list(Path(out_dir).glob(f"{vid}.*vtt")))
            # Run yt-dlp even if the file exists: it skips the existing caption
            # file but still prints the exact upload date.
            proc = run(
                f'yt-dlp --skip-download --no-simulate '
                f'--write-auto-sub --write-sub --sub-lang {LANG} --sub-format vtt '
                f'--print "%(upload_date)s" '
                f'-o "{out_dir}/%(id)s.%(ext)s" "https://www.youtube.com/watch?v={vid}"'
            )
            # Replace the approximate date with the exact one
            exact = proc.stdout.strip().splitlines()[-1:]
            if exact and re.fullmatch(r"\d{8}", exact[0]):
                videos.loc[videos["video_id"] == vid, "upload_date"] = exact[0]
            if had_file:
                skipped += 1
            elif proc.returncode == 0 and list(Path(out_dir).glob(f"{vid}.*vtt")):
                ok += 1
            else:
                failed += 1
            bar.set_postfix(ok=ok, skipped=skipped, failed=failed)
        totals["ok"] += ok; totals["skipped"] += skipped; totals["failed"] += failed

    # Re-group by the exact dates and drop videos older than the 10-year cutoff
    videos["year"] = videos["upload_date"].str[:4].astype(int)
    videos.drop(videos.index[videos["upload_date"] < CUTOFF], inplace=True)
    videos.to_csv(VIDEOS_CSV, index=False)
    print(f"\n{len(videos)} videos since {CUTOFF} (exact dates):")
    print(videos.groupby("year").size().to_string())
    print(f"\nFinished captions → new: {totals['ok']}, already had: {totals['skipped']}, "
          f"no captions/failed: {totals['failed']}")

download_captions_by_year(videos, OUT_DIR)



2026 captions:   0%|          | 0/45 [00:00<?, ?video/s]

2025 captions:   0%|          | 0/63 [00:00<?, ?video/s]

2024 captions:   0%|          | 0/58 [00:00<?, ?video/s]

2023 captions:   0%|          | 0/103 [00:00<?, ?video/s]

2022 captions:   0%|          | 0/69 [00:00<?, ?video/s]

2021 captions:   0%|          | 0/33 [00:00<?, ?video/s]

... (2 outputs left out)

2018 captions:   0%|          | 0/15 [00:00<?, ?video/s]

2017 captions:   0%|          | 0/28 [00:00<?, ?video/s]

2016 captions:   0%|          | 0/18 [00:00<?, ?video/s]


445 videos since 20161106 (exact dates):
year
2016     9
2017    25
2018    10
2019    17
2020    17
2021    38
2022    71
2023    98
2024    55
2025    62
2026    43

Finished captions → new: 359, already had: 30, no captions/failed: 74


In [7]:
##NEW SECTION 3: parse transcripts, one progress bar per year
TAG_RE = re.compile(r"<[^>]+>")   # strips <00:00:01.234><c> word </c> tags

def parse_vtt(vtt_file):
    """Returns [(timestamp, text)] with YouTube's rolling-caption repeats removed."""
    rows, timestamp = [], None
    recent = deque(maxlen=4)          # auto-captions repeat each line 2-3 times
    with open(vtt_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if "-->" in line:
                timestamp = line.split(" --> ")[0]
                continue
            if (not line or line.startswith(("WEBVTT", "Kind:", "Language:", "NOTE"))
                    or line.isdigit()):
                continue
            text = re.sub(r"\s+", " ", TAG_RE.sub("", line)).strip()
            if not text or text in recent:
                continue
            recent.append(text)
            rows.append((timestamp, text))
    return rows

def parse_by_year(videos, folder):
    all_rows = []
    for year, group in sorted(videos.groupby("year"), reverse=True):
        for row in tqdm(group.itertuples(), total=len(group),
                        desc=f"{year} parsing", unit="video"):
            files = sorted(Path(folder).glob(f"{row.video_id}.*vtt"))
            if not files:
                continue
            for ts, text in parse_vtt(files[0]):
                all_rows.append({"year": year, "upload_date": row.upload_date,
                                 "video_id": row.video_id, "title": row.title,
                                 "timestamp": ts, "text": text})
    df = pd.DataFrame(all_rows)
    df.to_csv(TRANSCRIPTS_CSV, index=False)
    print(f"\nSaved {TRANSCRIPTS_CSV}: {len(df)} lines from {df['video_id'].nunique()} videos")
    return df

transcripts = parse_by_year(videos, OUT_DIR)



2026 parsing:   0%|          | 0/43 [00:00<?, ?video/s]

2025 parsing:   0%|          | 0/62 [00:00<?, ?video/s]

2024 parsing:   0%|          | 0/55 [00:00<?, ?video/s]

2023 parsing:   0%|          | 0/98 [00:00<?, ?video/s]

2022 parsing:   0%|          | 0/71 [00:00<?, ?video/s]

2021 parsing:   0%|          | 0/38 [00:00<?, ?video/s]

... (2 outputs left out)

2018 parsing:   0%|          | 0/10 [00:00<?, ?video/s]

2017 parsing:   0%|          | 0/25 [00:00<?, ?video/s]

2016 parsing:   0%|          | 0/9 [00:00<?, ?video/s]


Saved all_transcripts.csv: 821030 lines from 378 videos


In [8]:
##NEW SECTION 4: search for "I am", one progress bar per year
INPUT_CSV  = "all_transcripts.csv"
OUTPUT_CSV = "i_am_mentions_ranked.csv"
BY_VIDEO_CSV = "i_am_by_video.csv"
INCLUDE_IM = False   # True also counts "I'm" / "I’m"
CONTEXT_CHARS = 60

PHRASE = r"\b(?:i\s+am|i['’]m)\b" if INCLUDE_IM else r"\bi\s+am\b"
PHRASE_RE = re.compile(PHRASE, re.IGNORECASE)

df = pd.read_csv(INPUT_CSV, dtype={"upload_date": str, "video_id": str})
df["text"] = df["text"].astype(str)

matches = []
for year, ydf in sorted(df.groupby("year"), reverse=True):
    vids = ydf.groupby("video_id", sort=False)
    for vid, vdf in tqdm(vids, total=vids.ngroups, desc=f"{year} searching", unit="video"):
        # Join the whole transcript so "I" at the end of one caption line and
        # "am" at the start of the next still count.
        starts, pos = [], 0
        for t in vdf["text"]:
            starts.append(pos); pos += len(t) + 1
        full = " ".join(vdf["text"])
        stamps = vdf["timestamp"].tolist()
        first = vdf.iloc[0]
        for m in PHRASE_RE.finditer(full):
            i = bisect.bisect_right(starts, m.start()) - 1
            a, b = max(0, m.start() - CONTEXT_CHARS), m.end() + CONTEXT_CHARS
            matches.append({
                "year": year, "upload_date": first["upload_date"],
                "video_id": vid, "title": first["title"],
                "video_url": f"https://www.youtube.com/watch?v={vid}",
                "timestamp": stamps[i],
                "context": ("…" if a else "") + full[a:b] + ("…" if b < len(full) else ""),
            })

res = pd.DataFrame(matches)
if res.empty:
    print("No mentions found.")
else:
    counts = res.groupby("video_id").size().rename("i_am_mentions")
    res = res.merge(counts, on="video_id")
    res["rank"] = res["i_am_mentions"].rank(method="dense", ascending=False).astype(int)
    res = res.sort_values(["i_am_mentions", "video_id", "timestamp"],
                          ascending=[False, True, True])
    res = res[["rank", "year", "upload_date", "video_id", "title", "video_url",
               "i_am_mentions", "timestamp", "context"]]
    res.to_csv(OUTPUT_CSV, index=False)

    by_video = (res.drop_duplicates("video_id")
                   [["rank", "year", "upload_date", "video_id", "title", "video_url", "i_am_mentions"]])
    by_video.to_csv(BY_VIDEO_CSV, index=False)

    print("Saved:", OUTPUT_CSV, "and", BY_VIDEO_CSV)
    print('Total "I am" mentions:', len(res))
    print('Videos containing "I am":', len(by_video))
    print("\nMentions per year:")
    print(res.groupby("year").agg(mentions=("video_id", "size"),
                                  videos=("video_id", "nunique"))
             .sort_index(ascending=False).to_string())

res.head(20)


2026 searching:   0%|          | 0/41 [00:00<?, ?video/s]

2025 searching:   0%|          | 0/56 [00:00<?, ?video/s]

2024 searching:   0%|          | 0/49 [00:00<?, ?video/s]

2023 searching:   0%|          | 0/83 [00:00<?, ?video/s]

2022 searching:   0%|          | 0/65 [00:00<?, ?video/s]

2021 searching:   0%|          | 0/31 [00:00<?, ?video/s]

... (3 outputs left out)

2017 searching:   0%|          | 0/15 [00:00<?, ?video/s]

2016 searching:   0%|          | 0/8 [00:00<?, ?video/s]

Saved: i_am_mentions_ranked.csv and i_am_by_video.csv
Total "I am" mentions: 2186
Videos containing "I am": 294

Mentions per year:
      mentions  videos
year                  
2026       437      40
2025       488      45
2024       285      42
2023       400      68
2022       423      58
2021       119      22
2020        13       7
2019        16       7
2017         5       5


,rank,year,upload_date,video_id,title,video_url,i_am_mentions,timestamp,context
1637,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:21:34.080,… name? What shall I say unto them? And God sa...
1638,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:21:34.080,…t shall I say unto them? And God said unto Mo...
1639,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:21:43.919,"…d he said, Thus shalt thou say unto the child..."
1640,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:41:40.240,…they are still demanding for a name. You let ...
1641,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:41:40.240,…till demanding for a name. You let them know ...
1642,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:41:57.040,"…again the second time and he said, ""You will ..."
1643,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:42:24.079,…an argument by introducing a more sophisticat...
1644,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:42:24.079,…t by introducing a more sophisticated what is...
1645,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:42:47.440,…ppeared and they don't understand that and th...
1646,1,2022,20220510,trz4-M5SZq4,THE SUPERNATURAL NATURAL SERIES | THE I AM,https://www.youtube.com/watch?v=trz4-M5SZq4,102,00:43:37.760,…t you are yet to acquire. What kind of Englis...


In [9]:
import shutil; from google.colab import files
files.download(shutil.make_archive("/content/i_am_outputs", "zip", "."))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>